# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Francesco Verni**: || SCIPER **428016**:


Person 2 **Davide Rossi**: || SCIPER **428412**:


Person 3 **Maguette Diouf**: || SCIPER **363572**:


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

We used Gemini to help solve doubts about the theory inherent to the questions in the notebook. It was useful to clarify specific questions which is often hard to find an answer to by simply going through the lecture materials.
The person in charge of this was Francesco Verni. The conversation is accessible at this [link](https://share.gemini.google/1SJBkyb9TrJX).


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



---

***Solution.***

We will generally denote the column of the class weight matrix corresponding to the outcome $b_i$ as $\mathbf{x}_{b_i}$. Now, we first derive the likelihood:

$$
\mathcal{L}(\mathbf{X})= \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n) = \prod_{i = 1}^n \mathbb{P}(b_i \mid \mathbf{a}_i) = \prod_{i = 1}^n \Bigg[ \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \Bigg].
$$

Taking the logarithm on both sides:

$$
\begin{align*}
\log \mathcal{L}(\mathbf{X}) = \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n) &= \sum_{i = 1}^n \log \Bigg[ \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \Bigg] = \\
&= \sum_{i = 1}^n \left[ \mathbf{a}_i^\top \mathbf{x}_{b_i}-\log\left( \sum_{k=1}^C \exp (\mathbf{a}_i^\top \mathbf{x}_k)\right)\right].
\end{align*}
$$

Negating this latter expression gives the desired result. $\quad \square$

---

__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

---

***Solution.***

We wish to show that $g: \mathbf{u} \mapsto \log \sum_{k=1}^{C} e^{u_{k}}, g: \mathbb{R}^{d} \rightarrow \mathbb{R}$ is convex on $\mathbb{R}^{C}$.

For a function $g$ to be convex on $\mathbb{R}^{C}$ we need
$$
g\left(\alpha \mathbf{u}_{1}+(1-\alpha) \mathbf{u}_{2}\right) \leq \alpha g\left(\mathbf{u}_{1}\right)+(1-\alpha) g\left(\mathbf{u}_{2}\right) \quad \forall \mathbf{u}_{1}, \mathbf{u}_{2} \in \mathbb{R}^{C}, \forall \alpha \in[0,1]
$$

Plugging in $g: \mathbf{u} \mapsto \log \sum_{k=1}^{C} e^{u_{k}}$:
$$
\log \sum_{k=1}^{a} \exp \left(\alpha u_{1, k}+(1-\alpha) u_{2, k}\right) \underset{?}{\leq} \alpha \log \sum \exp \left(u_{1, k}\right)+(1-\alpha) \log \sum \exp \left(u_{2, k}\right)
$$

Exponentiating both sides,
$$
\begin{align*}
 \sum_{k} \exp \left(\alpha u_{1, k}+(1-\alpha) u_{2, k}\right) &\underset{?}{\leq} \exp \left\{\log \left[\sum \exp \left(u_{1, k}\right)\right]^{\alpha}+(1-\alpha) \log \left[\sum \exp \left(u_{2, k}\right)\right]^{1-\alpha}\right\} \\
 \sum_{k} \exp \left(\alpha u_{1, k}\right) \exp \left((1-\alpha) u_{2, k}\right) &\underset{?}{\leq}\left[\sum \exp \left(u_{1, k}\right)\right]^{\alpha}\left[\sum \exp \left(u_{2, k}\right)\right]^{1-\alpha} \\
\sum_{k} \frac{\exp \left(\alpha u_{1, k}\right)}{\left[\sum \exp \left(u_{1, k}\right)\right]^{\alpha}} \cdot \frac{\exp \left((1-\alpha) u_{2, k}\right)}{\left[\sum \exp \left(u_{2, k}\right)\right]^{1-\alpha}} &\underset{?}{\leq} 1.
\end{align*}
$$

Recognising the Softmax pmf we get

$$
\begin{align*}
\sum_{k} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)^{\alpha} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{2}\right)^{1-\alpha} &\underset{?}{\leq} 1 \\
\sum_{k} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)^{\alpha} \frac{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)^{1-\alpha}}{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)^{1-\alpha}} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{2}\right)^{1-\alpha} &\underset{?}{\leq}1 \\
\sum_{k} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right) \frac{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{2}\right)^{1-\alpha}}{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1})^{1-\alpha}\right.} &\underset{?}{\leq} 1 \\
E_{b_{i} \mid \mathbf{u}_{1}}\left\{\left[\frac{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{1}\right)}\right]^{1-\alpha}\right\} &\underset{?}{\leq} 1
\end{align*}
$$
applying the definition of expected value for discrete random variables.

By Jensen’s inequality $E_{b_{i} \mid \mathbf{u}_{1}}\left\{\left[\frac{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{1}\right)}\right]^{1-\alpha}\right\} \leq\left\{E_{b_{i} \mid \mathbf{u}_{1}} \left[\frac{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{1}\right)}\right]\right\}^{1-\alpha}$.

This is because $y=x^{\theta}$ is concave by the definition of convex function using the positive semi-definiteness of the Hessian (which here reduces to a scalar):

$$
\frac{\partial}{\partial x} y=\theta x^{\theta-1}, \quad \frac{\partial^{2}}{\partial x^{2}} y=\theta(\theta-1) x^{\theta-2} \leqslant 0 \quad \text { for any } \theta \in[0,1]
$$

The RHS of the Jensen inequality we enunciated above is
$$
\begin{align*}
\left\{E_{b_{i} \mid \mathbf{u}_{1}}\left[\frac{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{1}\right)}\right]\right\}^{1-\alpha}= & \sum_{k=1}^{c} \frac{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{1}\right)= \\
& =\sum_{k} \mathbb{P}\left(b_{i}=k \mid \mathbf{u}_{2}\right)=1 \quad \text { since it is a pmf.}
\end{align*}
$$

This implies that

$$
E_{b_{i} \mid \mathbf{u}_{1}}\left\{\left[\frac{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{2}\right)}{\mathbb{P}\left(b_{i} \mid \mathbf{u}_{1}\right)}\right]^{1-\alpha}\right\} \leq 1. \quad \square
$$

---

Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


---

***Solution.***

Let $f: \mathcal{X} \to \mathbb{R}$ be a function defined on a domain $\mathcal{X}$. The infimum of $f$, denoted $\inf_{x \in \mathcal{X}} f(x)$, is the greatest lower bound of its image $f(\mathcal{X})$. Formally, a real number $m = \inf_{x \in \mathcal{X}} f(x)$ if $m \leq f(x)$ for all $x \in \mathcal{X}$, and for every $\epsilon > 0$, there exists $x \in \mathcal{X}$ such that $f(x) < m + \epsilon$. A minimum is an infimum that is explicitly attained by the function within its domain. Specifically, the infimum is a minimum, denoted $\min_{x \in \mathcal{X}} f(x)$, if and only if $\exists x^* \in \mathcal{X}$ such that $f(x^*) = \inf_{x \in \mathcal{X}} f(x)$. Consequently, all minima are infima, but an infimum is a minimum strictly when the infimum is an element of the image set $f(\mathcal{X})$.

To illustrate a convex function on $\mathbb{R}$ that does not attain its infimum, consider $f: \mathbb{R} \to \mathbb{R}$ defined by $f(x) = e^x$. This function is strictly convex and bounded below by $0$. Evaluating the asymptotic behavior yields $\lim_{x \to -\infty} e^x = 0$, establishing that $\inf_{x \in \mathbb{R}} e^x = 0$. However, because $\nexists x \in \mathbb{R}$ such that $e^x = 0$, the infimum is not attained in the domain, and therefore $\min_{x \in \mathbb{R}} e^x$ is undefined.

---

__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


---

***Solution.***

*Geometric interpretation.* For $p=2$, the decision boundary becomes a straight line. For a single comparison between one class $c$ and a competing class $k \neq c$ the linear decision boundary is 

$$
\mathbf{a}_i^\top (\mathbf{x}_{0,c}-\mathbf{x}_{0, k}) = 0 \iff (x_{0,c,1} - x_{0,k, 1})a_{i,1} + (x_{0,c,2} - x_{0,k, 2})a_{i, 2} = 0.
$$

This results in a different linear decision boundary for discriminating each pair of classes $(c, k)$ which is based on the difference between the parameter vectors corresponding to the two competing classes. These parameter vectors can be interpreted as the direction that the model assigns to a specific class, along which the score $\langle \mathbf{a}_i, \mathbf{x}_{0,c} \rangle$ grows more rapidly in function of $\mathbf{a}_i$, and therefore the boundary line will be the straight line perpendicular to the chord (i.e. the line with the same direction as the difference vector) connecting two of these parameter vectors. The inner product $\langle \mathbf{a}_i, \mathbf{x}_{0,c} \rangle = \| \mathbf{a}_i \|_2 \| \mathbf{x}_{0,c} \|_2 \cos\theta $ will then measure the "degree of alignment" (more precisely the resulting score considers the norms of the vectors as well) between the feature vector (the vector $\mathbf{a}_i$, representing the coordinates of the $i$-th observation in the 2-dimensional feature space) and the class-specific parameter vector. One-versus-all complete separation guarantees that this inner product score with respect to a particular observation $i$ achieves its global maximum when the inner product is computed using the parameter vector corresponding to $i$'s true class, i.e. $b_i$. 
This is a global maximum because no datapoint can lie on the boundary between two classes (in the two dimensional case). This translates, in higher dimensional spaces, into the guarantee that the feature space can be exactly partitioned into "class-specific" regions. This means that between any two competing classes $c$ and $k$ there will always exist a hyperplane which separates the region in which the inner product score is maximised by $\mathbf{x}_c$ and the region in which it is maximised by $\mathbf{x}_k$. Moreover, these hyperplanes do not contain any of the observed datapoints.

The name *one-versus-all complete separation* is appropriate because the inequality mathematically compares the score of the correct class (*one*) against the maximum score of all alternative classes (*all*). Moreover, since the inequality holds for every $i$, we have a *complete* separation because no points lie on the boundary hyperplanes, hence if we simply were to predict each class by choosing the maximiser of the scores $\arg\max_{k=1,\ldots, C} (\mathbf{a}_i^\top\mathbf{x}_{0,k})$ we would predict the true class label. 

---

Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


---
***Solution.***

By assumption, $\mathbf{X}_0$ achieves complete separation, that is, for all $i$,
$$\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.$$
It is straightforward to check that if $\mathbf{X}_0$ achieves complete separation, then $\alpha \mathbf{X}_0$ achieves complete separation as well for any $\alpha > 0$:

$$
\mathbf{a}_i^\top (\alpha\mathbf{x}_{0, b_i}) - \max_{k \neq b_i} \mathbf{a}_i^\top (\alpha\mathbf{x}_{0,k}) > 0 \iff \alpha \left(\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k}\right) > 0 \iff \mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$

To evaluate the behavior of the negative log-likelihood under complete separation, we substitute the scaled weight matrix $\alpha \mathbf{X}_0$ into the objective function:
$$f(\alpha \mathbf{X}_0) = \sum_{i=1}^n \left[ -\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i} + \log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0,k}) \right].$$

We can factor out the exponential term corresponding to the true class $b_i$ from the sum inside the logarithm:
$$\sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0,k}) = \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}) \left( 1 + \sum_{k \neq b_i} \exp\big(\alpha (\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})\big) \right),$$

hence retaking the logarithm

$$\log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0,k}) = \alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i} + \log \left( 1 + \sum_{k \neq b_i} \exp\big(\alpha (\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})\big) \right).$$

Substituting this back into $f(\alpha \mathbf{X}_0)$, the $-\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}$ and $+\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}$ terms cancel out, yielding:
$$f(\alpha \mathbf{X}_0) = \sum_{i=1}^n \log \left( 1 + \sum_{k \neq b_i} \exp\big(\alpha (\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})\big) \right).$$

The complete separation assumption guarantees that $\alpha\mathbf{a}_i^\top \mathbf{x}_{0,b_i} > \alpha\mathbf{a}_i^\top \mathbf{x}_{0,k}$ for all $k \neq b_i$. Consequently, the exponent is strictly negative for all incorrect classes:
$$\alpha(\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i}) < 0.$$

As $\alpha \to +\infty$, the exponential of a strictly negative number approaches zero:
$$\lim_{\alpha \to +\infty} \exp\big(\alpha (\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})\big) = 0.$$

Evaluating the limit for the entire function gives:
$$\lim_{\alpha \to +\infty} f(\alpha \mathbf{X}_0) = \sum_{i=1}^n \log(1 + 0) = 0.$$
This shows that the infimum of $f(\mathbf{X})$ is $0$. 

However, for any finite weight matrix $\mathbf{X} \in \mathbb{R}^{p \times C}$, the sum $\sum_{k \neq b_i} \exp(\mathbf{a}_i^\top \mathbf{x}_k - \mathbf{a}_i^\top \mathbf{x}_{b_i})$ is strictly greater than $0$ because the exponential function never evaluates to zero. Therefore, $f(\mathbf{X}) > 0$ $\forall \mathbf{X} \in \mathbb{R}^{p \times C}$. Because the function can approach $0$ asymptotically but never reach it for any finite parameters, it does not attain its minimum.  $\quad$ $\square$

---

We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



---

***Solution.***

By the linearity of the gradient operator, we can separate the derivative of the regularised objective function $f_\mu(\mathbf{X})$ into the gradient of the negative log-likelihood $f(\mathbf{X})$ and the gradient of the regularisation term, yielding the equation:

$$\nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \nabla_{\mathbf{X}} f(\mathbf{X}) + \nabla_{\mathbf{X}} \left( \frac{\mu}{2} \Vert{}\mathbf{X}\Vert{}_F^2 \right)$$

By definition, the Frobenius norm of a matrix $\mathbf{X} \in \mathbb{R}^{p \times C}$ is given by 

$$\|\mathbf{X}\|_F = \sqrt{\sum_{i=1}^p \sum_{j=1}^C X_{ij}^2},$$

or equivalently as

$$\Vert\mathbf{X}\Vert_F = \sqrt{\text{Tr}(\mathbf{X}^\top \mathbf{X})}.$$

So:

$$\Vert\mathbf{X}\Vert_F^2 = \text{Tr}(\mathbf{X}^\top \mathbf{X})$$

Therefore, the gradient of the regularisation term evaluates to

$$\nabla_{\mathbf{X}} \left( \frac{\mu}{2} \|\mathbf{X}\|_F^2 \right) = \frac{\mu}{2}  \text{Tr}(\nabla_{\mathbf{X}}\mathbf{X}^\top \mathbf{X}) = \mu \mathbf{X},$$

where $\mathbf{X}$ is composed of $C$ column vectors $\mathbf{x}_c \in \mathbb{R}^p$:

$$\mathbf{X} = \begin{bmatrix} \mathbf{x}_1 & \mathbf{x}_2 & \dots & \mathbf{x}_C \end{bmatrix}.$$

As showed in point **(a)** the negative log-likelihood is given by

$$f(\mathbf{X}) = \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right],$$

so that we can define

$$f(\mathbf{X}) = \sum_{i=1}^n f_i(\mathbf{X}),$$

where

$$f_i(\mathbf{X}) = -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k).$$

The full gradient matrix $\nabla_{\mathbf{X}} f(\mathbf{X})$ is therefore given by

$$\nabla_{\mathbf{X}} f(\mathbf{X}) = \begin{bmatrix} \frac{\partial f}{\partial \mathbf{x}_1} & \frac{\partial f}{\partial \mathbf{x}_2} & \dots & \frac{\partial f}{\partial \mathbf{x}_C} \end{bmatrix},$$

where each component $\frac{\partial f}{\partial \mathbf{x}_c}$ is a $p$ dimensional column vector containing the partial derivatives of $f$ with respect to each scalar element of $\mathbf{x}_c$

$$\frac{\partial f}{\partial \mathbf{x}_c} = \begin{bmatrix} \frac{\partial f}{\partial x_{1,c}} \\ \frac{\partial f}{\partial x_{2,c}} \\ \vdots \\ \frac{\partial f}{\partial x_{p,c}} \end{bmatrix}.$$

The derivative of $-\mathbf{a}_i^\top \mathbf{x}_{b_i}$ with respect to $\mathbf{x}_c$ is $-\mathbf{a}_i$ if the target class $b_i$ is equal to $c$, and 0 otherwise. We can write this compactly using the the one-hot vector $\mathbf{e}_{b_i}$, giving:

$$\frac{\partial}{\partial \mathbf{x}_c} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} \right) = \mathbb{I}(b_i = c)\cdot(-\mathbf{a}_i) = -e_{b_i,c} \mathbf{a}_i.$$

Using the  the chain rule to differentiate the summation, since the only term in the sum that depends on $\mathbf{x}_c$ is $\exp(\mathbf{a}_i^\top \mathbf{x}_c)$, the derivative becomes:

$$\frac{\partial}{\partial \mathbf{x}_c} \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) = \frac{1}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \cdot \exp(\mathbf{a}_i^\top \mathbf{x}_c) \cdot \mathbf{a}_i.$$

Therefore:

$$\frac{\partial}{\partial \mathbf{x}_c} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) = -e_{b_i,c} \mathbf{a}_i + \left( \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \right) \mathbf{a}_i.$$

Note that the softmax probability for class $c$, denoted as $p_{i,c}$ is given by
$$p_{i,c} = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.$$

Substituting $p_{i,c}$ back into the equation allows us to simplify the final gradient for sample $i$ with respect to $\mathbf{x}_c$:

$$\frac{\partial}{\partial \mathbf{x}_c} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) = p_{i,c} \mathbf{a}_i - e_{b_i,c} \mathbf{a}_i = \mathbf{a}_i (p_{i,c} - e_{b_i,c}).$$

To assemble the gradient matrix for a single sample $i$ with respect to the entire matrix $\mathbf{X}$, we concatenate the column gradients for all classes $c \in \{1, \dots, C\}$, that is:

$$\nabla_{\mathbf{X}} f_i(\mathbf{X}) = \begin{bmatrix} \frac{\partial}{\partial \mathbf{x}_1} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) & \dots & \frac{\partial}{\partial \mathbf{x}_C} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \end{bmatrix}.$$

Substituting the simplified column derivatives derived earlier, this evaluates to:
$$\nabla_{\mathbf{X}} f_i(\mathbf{X}) = \begin{bmatrix} \mathbf{a}_i (p_{i,1} - e_{b_i,1}) & \mathbf{a}_i (p_{i,2} - e_{b_i,2}) & \dots & \mathbf{a}_i (p_{i,C} - e_{b_i,C}) \end{bmatrix}= \mathbf{a}_i \begin{bmatrix} (p_{i,1} - e_{b_i,1}) & (p_{i,2} - e_{b_i,2}) & \dots & (p_{i,C} - e_{b_i,C}) \end{bmatrix}=\mathbf{a}_i (\mathbf{p}_i - \mathbf{e}_{b_i})^\top.$$

By linearity of the derivative, we sum the gradient for the single sample $\nabla_{\mathbf{X}} f_i(\mathbf{X})$ over all $n$ independent samples to obtain the full gradient of the negative log-likelihood $f(\mathbf{X})$:

$$\nabla_{\mathbf{X}} f(\mathbf{X}) = \sum_{i=1}^n \nabla_{\mathbf{X}} f_i(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i (\mathbf{p}_i - \mathbf{e}_{b_i})^\top.$$

Next, we recall that the full regularised objective function $f_\mu(\mathbf{X})$ is the sum of this negative log-likelihood and the Frobenius norm regularization term

$$f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \Vert\mathbf{X}\Vert_F^2,$$

and applying the gradient operator with respect to $\mathbf{X}$ to both sides gives:

$$
\begin{align*}
\nabla_{\mathbf{X}} f_\mu(\mathbf{X}) &= \nabla_{\mathbf{X}} f(\mathbf{X}) + \nabla_{\mathbf{X}} \left( \frac{\mu}{2} \Vert\mathbf{X}\Vert_F^2 \right)=\\
&= \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X}. \quad \square
\end{align*} 
$$

---

__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$

where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


---

***Solution.***

Let $\mathbf{X} \in \mathbb{R}^{p\times C}$ and $\tilde{\mathbf{x}} = \operatorname{vec}(\mathbf{X})\in \mathbb{R}^{pC}$ be the row-wise vectorisation of $\mathbf{X}$. Defining $\tilde{f}_\mu(\tilde{\mathbf{x}}) := f_\mu(\operatorname{mat}({\mathbf{\tilde{x}}}))$, where $\operatorname{mat}:= \operatorname{vec}^{-1}$, we then aim to evaluate 

$$
\nabla^2 f_\mu(\mathbf{X}) = \nabla^2 f_\mu(\operatorname{mat}(\tilde{\mathbf{x}})) = \nabla^2\tilde{f}_\mu(\tilde{\mathbf{x}}). 
$$

By linearity of the Hessian operator:

$$
\nabla^2 f_\mu(\mathbf{X}) = \nabla^2 f(\mathbf{X}) + \nabla^2 \left( \frac{\mu}{2} \Vert\mathbf{X}\Vert_F^2 \right).
$$

The regulariser satisfies 

$$
\frac{\mu}{2} \Vert\mathbf{X}\Vert_F^2 = \frac{\mu}{2} \Vert\operatorname{mat}(\tilde{\mathbf{x}})\Vert_F^2 = \frac{\mu}{2}\sum_{i=1}^p \sum_{j=1}^C X_{ij}^2 = \frac{\mu}{2} \| \tilde{\mathbf{x}}\|_2^2,
$$

hence 

$$
\nabla_{\mathbf{X}}^2 \left( \frac{\mu}{2} \Vert\mathbf{X}\Vert_F^2 \right) = \nabla_{\tilde{\mathbf{x}}}^2 \left(\frac{\mu}{2} \tilde{\mathbf{x}}^\top\tilde{\mathbf{x}}\right) = \nabla_{\tilde{\mathbf{x}}} (\mu \tilde{\mathbf{x}}) = \mu \mathbf{I}_{pC}.
$$

To derive the $pC \times pC$ Hessian matrix using only standard partial derivatives, we compute the second derivative of the negative log-likelihood for a single sample $i$ with respect to individual scalar elements of $\mathbf{X}$, which directly map to the elements of its row-wise vecotrised form $\tilde{\mathbf{x}}$. Let $X_{jc}$ be the scalar element in the $j$-th row and $c$-th column of $\mathbf{X}$. From point **(f)**:

$$
f(\mathbf{X}) = \sum_{i=1}^n f_i(\mathbf{X}) \quad \text{where} \quad f_i(\mathbf{X}) = -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k).
$$

Then the partial derivative of the negative log-likelihood $f_i$ with respect to $X_{jc}$ is:

$$
\frac{\partial f_i}{\partial X_{jc}} = a_{i,j} (p_{i,c} - e_{b_i,c}),
$$

where $a_{i,j}$ is the $j$-th element of the feature vector $\mathbf{a}_i$.

To find the elements of the Hessian matrix, we take the second partial derivative with respect to another arbitrary element $X_{kd}$:
$$\frac{\partial^2 f_i}{\partial X_{jc} \partial X_{kd}} = \frac{\partial}{\partial X_{kd}} \big[ a_{i,j} (p_{i,c} - e_{b_i,c}) \big] = a_{i,j} \frac{\partial p_{i,c}}{\partial X_{kd}}.$$

By the chain rule, we can differentiate $p_{i,c}$ with respect to $X_{kd}$:

$$\frac{\partial p_{i,c}}{\partial X_{kd}} = \frac{\partial}{\partial X_{kd}}\frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k'=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_{k'})}=\sum_{m=1}^C \frac{\partial p_{i,c}}{\partial z_{i,m}} \frac{\partial z_{i,m}}{\partial X_{kd}},$$

where $z_{i,m} = \mathbf{a}_i^\top \mathbf{x}_m = \sum_{l=1}^p a_{i,l} X_{l,m}$. Notice that $z_{i,m}$ only depends on the elements in the $m$-th column of the matrix $\mathbf{X}$. But we are differentiating with respect to $X_{kd}$, which is an element in the $d$-th column of $\mathbf{X}$.

So, if $m \neq d$, the formula for $z_{i,m}$ does not contain the variable $X_{kd}$. Therefore, $\frac{\partial z_{i,m}}{\partial X_{kd}} = 0$. If $m = d$, the formula is $z_{i,d} = \sum_{l=1}^p a_{i,l} X_{l,d}$ and its derivative with respect to $X_{kd}$ is the coefficient $a_{i,k}$.

Because of this:

$$
\frac{\partial p_{i,c}}{\partial X_{kd}} = \sum_{m=1}^C \frac{\partial p_{i,c}}{\partial z_{i,m}} \frac{\partial z_{i,m}}{\partial X_{kd}} = \frac{\partial p_{i,c}}{\partial z_{i,1}}(0) + \dots + \frac{\partial p_{i,c}}{\partial z_{i,d}}(a_{i,k}) + \dots + \frac{\partial p_{i,c}}{\partial z_{i,C}}(0)=\frac{\partial p_{i,c}}{\partial z_{i,d}}a_{i,k},
$$

Now we need  to evaluate $\frac{\partial p_{i,c}}{\partial z_{i,d}}$. Let $c = d$. Differentiating with respect to $z_{i,c}$ directly transforms the quotient rule application into the factored probability:

$$
\frac{\partial p_{i,c}}{\partial z_{i,c}} = \frac{\exp(z_{i,c}) \sum_{m=1}^C \exp(z_{i,m}) - \exp(z_{i,c})^2}{\left( \sum_{m=1}^C \exp(z_{i,m}) \right)^2} = p_{i,c}(1 - p_{i,c}).
$$

Now let $c \neq d$ instead. Differentiating with respect to $z_{i,d}$ zeroes out the numerator derivative, reducing the quotient application directly to the negative product of the probabilities:
$$\frac{\partial p_{i,c}}{\partial z_{i,d}} = \frac{0 - \exp(z_{i,c})\exp(z_{i,d})}{\left( \sum_{m=1}^C \exp(z_{i,m}) \right)^2} = -p_{i,c} p_{i,d}.$$

Combining the two cases

$$
\frac{\partial p_{i,c}}{\partial z_{i,d}} = p_{i,c}(\delta_{cd} - p_{i,d}) \quad \text {where} \quad \delta_{cd} = \begin{cases}1, \quad\text {if } c=d\\0, \quad\text{otherwise}
\end{cases}
$$

and plugging this back into the expression derived earlier we get:

$$\frac{\partial p_{i,c}}{\partial X_{kd}} = p_{i,c} (\delta_{cd} - p_{i,d}) a_{i,k}.$$

Inserting this into our second derivative equation, we obtain the scalar element of the Hessian:
$$\frac{\partial^2 f_i}{\partial X_{jc} \partial X_{kd}} = a_{i,j} a_{i,k} p_{i,c} (\delta_{cd} - p_{i,d}).$$

This scalar value is the product of two distinct terms:

**(i)** $a_{i,j} a_{i,k}$, which is the $(j, k)$-th element of the $p \times p$ outer product matrix $\mathbf{a}_i \mathbf{a}_i^\top$;
  
**(ii)** $p_{i,c} (\delta_{cd} - p_{i,d})$, which forms the $(c, d)$-th entry of the $C \times C$ softmax Jacobian matrix $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$. This structure is immediately clear by expanding the matrix expression:$$\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top = \begin{bmatrix} p_{i,1} & 0 & \dots & 0 \\ 0 & p_{i,2} & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & p_{i,C} \end{bmatrix} - \begin{bmatrix} p_{i,1}^2 & p_{i,1}p_{i,2} & \dots & p_{i,1}p_{i,C} \\ p_{i,2}p_{i,1} & p_{i,2}^2 & \dots & p_{i,2}p_{i,C} \\ \vdots & \vdots & \ddots & \vdots \\ p_{i,C}p_{i,1} & p_{i,C}p_{i,2} & \dots & p_{i,C}^2 \end{bmatrix}.$$


Therefore, mapping all scalar second derivatives $\frac{\partial^2 f_i}{\partial X_{jc} \partial X_{kd}}$ into a $pC \times pC$ matrix using the Kronecker product yields:

$$
\nabla^2 \tilde{f}_i(\mathbf{X}) = (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big).
$$

By linearity of the Hessian operator, we sum this over all $n$ samples to get the Hessian of the negative log-likelihood:

$$
\nabla^2 \tilde{f}(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big).
$$

Finally, adding the Hessian of the regularisation term gives the complete regularised Hessian:

$$
\nabla^2 \tilde{f}_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC}.\quad \square
$$ 

---

__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


---

***Solution.***

By the lemma seen in class, a twice-differentiable convex function $\tilde{f}: \mathcal{Q} \to \mathbb{R}$, with $\mathcal{Q} \subseteq \mathbb{R}^{pC}$, i.e. $\tilde{f} \in \mathcal{F}^2(\mathcal{Q})$, is $\mu$-strongly convex if and only if its Hessian satisfies $\nabla^2 \tilde{f}(\mathbf{x}) \succeq \mu \mathbf{I}$ for all $\mathbf{x} \in \mathcal{Q}$. 

From part **(g)**, the Hessian of the regularised objective function is given by:

$$
\nabla^2 \tilde{f}_\mu(\tilde{\mathbf{x}}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC}.
$$

The outer product matrix $\mathbf{a}_i\mathbf{a}_i^\top$ is positive semidefinite because for any arbitrary vector $\mathbf{v}$, the quadratic form yields:

$$
\mathbf{v}^\top (\mathbf{a}_i\mathbf{a}_i^\top) \mathbf{v} = (\mathbf{a}_i^\top \mathbf{v})^2 \geq 0.
$$

Now, in order to establish whether $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is positive semidefinite we can derive that it represents the variance-covariance matrix of a multinomial distribution that is $\boldsymbol{\Sigma} = \operatorname{Var}(\mathbf{e}_{b_i})$,  and is therefore a positive semidefinite matrix. This can be seen in the following way.

We represent the target label as a random vector $\mathbf{e}_{b_i} \in \mathbb{R}^C$, where $\mathbf{e}_{b_i}$ is the one-hot vector for class $b_i$. For any given sample, exactly one element of $\mathbf{e}_{b_i}$ is $1$ and the remaining $C - 1$ elements are $0$. Let $e_{i,c}$ denote the $c$-th component of this vector, meaning $e_{i,c} \in \{0, 1\}$. 
Under the softmax model, $\mathbf{e}_{b_i}$ follows a multinomial distribution with a single trial and probability vector $\mathbf{p}_i$. The expected value of any component $e_{i,c}$ is the probability that $b_i = c$:

$$
\mathbb{E}[e_{i,c}] = 1 \cdot p_{i,c} + 0 \cdot (1 - p_{i,c}) = p_{i,c}.
$$

Thus, the mean vector is $\mathbb{E}[\mathbf{e}_{b_i}] = \mathbf{p}_i$. The variance-covariance matrix $\boldsymbol{\Sigma} = \operatorname{Var}(\mathbf{e}_{b_i})$ is populated by the variances of each component on the diagonal and the covariances between components on the off-diagonals. For the diagonal elements:

$$
\operatorname{Var}(e_{i,c}) = \mathbb{E}[e_{i,c}^2] - (\mathbb{E}[e_{i,c}])^2 = p_{i,c} - p_{i,c}^2 = p_{i,c}(1 - p_{i,c}).
$$

Considering the off-diagonal elements for any two distinct classes $c \neq k$, the covariance is:

$$
\operatorname{Cov}(e_{i,c}, e_{i,k}) = \mathbb{E}[e_{i,c} e_{i,k}] - \mathbb{E}[e_{i,c}]\mathbb{E}[e_{i,k}].
$$

Because the label $b_i$ can only take one class value at a time, the vector $\mathbf{e}_{b_i}$ cannot have ones in both position $c$ and position $k$ simultaneously. This means their product $e_{i,c} e_{i,k}$ is strictly $0$ in any case in which $c \neq k$. Therefore, $\mathbb{E}[e_{i,c} e_{i,k}] = 0$, and

$$
\operatorname{Cov}(e_{i,c}, e_{i,k}) = 0 - p_{i,c}p_{i,k} = -p_{i,c}p_{i,k}.
$$

Wrapping up we can express the whole matrix as 

$$
\boldsymbol{\Sigma} = \operatorname{Var}(\mathbf{e}_{b_i})= \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top,
$$

thus completing the proof that $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \succeq 0.$ 

A property of the Kronecker product is that the Kronecker product of two positive semidefinite matrices is positive semidefinite. Consequently:

$$
(\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) \succeq 0.
$$

Because the sum of positive semidefinite matrices is also positive semidefinite, summing over all $n$ samples guarantees that the entire unregularised Hessian $\nabla^2 \tilde{f}(\tilde{\mathbf{x}})$ is positive semidefinite ($\nabla^2 \tilde{f}(\tilde{\mathbf{x}}) \succeq 0$). Finally, plugging this result in the expression of the regularised Hessian:

$$
\nabla^2 {f}_\mu({\mathbf{X}}) = \nabla^2 \tilde{f}_\mu(\tilde{\mathbf{x}}) = \nabla^2 \tilde{f}(\tilde{\mathbf{x}}) + \mu \mathbf{I}_{pC} \succeq \mu \mathbf{I}_{pC}. \quad \square
$$

---

__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).

***Solution.***

It is not possible for a strongly convex function to not attain its minimum. We can verify this as follows. 

We assume for simplicity that $f$ is continuously differentiable. Recalling the quadratic lower bound for $\mu$-strongly convex functions, we have that a continuously differentiable function $f: \mathbb{R}^{p\times C} \to \mathbb{R}$ belongs to $\mathcal{F}_\mu^1(\mathbb{R}^{p\times C})$ if there exists a constant $\mu>0$ such that for any $\mathbf{X}, \mathbf{Y} \in \mathbb{R}^{p\times C}$: 

$$
f(\mathbf{Y}) \geq f(\mathbf{X}) + \langle \nabla f(\mathbf{X}), \mathbf{Y} - \mathbf{X} \rangle + \frac{\mu}{2}\|\mathbf{Y}-\mathbf{X}\|^2_F.
$$

Now consider the sublevel set $S = \{\mathbf{Y} \in \mathbb{R}^{p \times C} : f(\mathbf{Y}) \le f(\mathbf{X})\} = f^{-1}\big((-\infty,f(\mathbf X)]\big)$. For any $\mathbf{Y} \in S$: 

$$
f(\mathbf{X}) \ge f(\mathbf{Y}) \ge f(\mathbf{X}) + \langle \nabla f(\mathbf{X}), \mathbf{Y} - \mathbf{X} \rangle + \frac{\mu}{2}\|\mathbf{Y}-\mathbf{X}\|^2_F,
$$

implying that 

$$
\langle \nabla f(\mathbf{X}), \mathbf{Y} - \mathbf{X} \rangle + \frac{\mu}{2}\|\mathbf{Y}-\mathbf{X}\|^2_F \le 0.
$$

Applying Cauchy-Schwarz inequality to the inner product, we get that 

$$
\frac{\mu}{2}\|\mathbf{Y} - \mathbf{X}\|_F^2 \le -\langle \nabla f(\mathbf{X}), \mathbf{Y} - \mathbf{X} \rangle \le \|\nabla f(\mathbf{X})\|_F \|\mathbf{Y} - \mathbf{X}\|_F
$$

$$
\frac{\mu}{2}\|\mathbf{Y} - \mathbf{X}\|_F^2 \le \|\nabla f(\mathbf{X})\|_F \|\mathbf{Y} - \mathbf{X}\|_F
$$

If we now divide both sides by $\frac{\mu}{2}\|\mathbf{Y} - \mathbf{X}\|_F$, we obtain that

$$
\|\mathbf{Y} - \mathbf{X}\|_F \le \frac{2}{\mu}\|\nabla f(\mathbf{X})\|_F = r.
$$

This shows that the set $S$ of points with $f(\mathbf{Y}) \le f(\mathbf{X})$ is bounded inside a finite ball of radius $r = \frac{2}{\mu}\|\nabla f(\mathbf{X})\|_F$. Moving infinitely far away forces $f(\mathbf{Y}) > f(\mathbf{X})$, so a minimum cannot "escape" to infinity. Notice that in the case in which $\mathbf{Y} - \mathbf{X} = 0$, the inequality holds true trivially and we still have that S is bounded.

Since $f$ is assumed to be a continuously differentiable (and hence continuous), its sublevel set $S$ is closed (it is the preimage of a closed set) in finite dimensions ($\mathbb{R}^{p \times C}$), by Heine-Borel theorem any set that is both closed and bounded is compact. By the Weierstrass theorem, the continuous function $f$ restricted to the non-empty compact set $S$ must attain a minimum at some point $\mathbf{Y}^\star \in S$. Since any point outside $S$ has $f(\mathbf{Y}) > f(\mathbf{X}) \ge f(\mathbf{Y}^\star)$, $\mathbf{Y}^\star$ is a global minimiser over all of $\mathbb{R}^{p \times C}$.

Moreover, not only this minimiser exists, but it is also unique. This follows directly from the fact that a $\mu$-strongly convex function is necessarily strictly convex, and a strictly convex function has at most one minimiser.

This is something we can easily prove by contradiction. Suppose there were two distinct minimisers $\mathbf{X}_1^\star \neq \mathbf{X}_2^\star$ with $f(\mathbf{X}_1^\star) = f(\mathbf{X}_2^\star) = f^\star$. Evaluating $f$ at their midpoint (i.e. letting $\alpha = 1/2$ in the definition of strict convexity), 

$$
f\left(\frac{1}{2}\mathbf{X}_1^\star + \frac{1}{2}\mathbf{X}_2^\star\right) < \frac{1}{2}f(\mathbf{X}_1^\star) + \frac{1}{2}f(\mathbf{X}_2^\star) = f^\star
$$

which contradicts $f^\star$ being the global minimum value. $\quad \square$

---

We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.

---

***Solution.***

We aim to show that the largest eigenvalue of the matrix $\mathbf{a}_i\mathbf{a}_i^\top$ is equal to the squared $\text{L}_2$-norm of $\mathbf{a}_i$.

Since $\mathbf{a}_i\mathbf{a}_i^\top = (\mathbf{a}_i\mathbf{a}_i^\top)^\top$ by construction we can use the spectral decomposition of $\mathbf{a}_i\mathbf{a}_i^\top$ as 

$$
\mathbf{a}_i\mathbf{a}_i^\top = \mathbf{U}\mathbf{\Lambda}\mathbf{U}^\top, \quad \text{where} \quad \mathbf{\Lambda} = \operatorname{Diag}(\lambda_1, \ldots, \lambda_p).
$$

In particular, since the eigenvalues are ordered according to $\lambda_1 \ge \ldots \ge \lambda_p \ge 0$ (because $\mathbf{a}_i\mathbf{a}_i^\top \succeq 0$, as showed in **(h)**), we have that $\lambda_{\text{max}}(\mathbf{a}_i\mathbf{a}_i^\top) = \lambda_1$. Any eigenvalue of $\mathbf{a}_i\mathbf{a}_i^\top$ has to satisfy the condition 

$$
\mathbf{a}_i\mathbf{a}_i^\top \mathbf{v}= \lambda \mathbf{v},
$$ 

and noticing that if we let $\mathbf{v} = \mathbf{a}_i$, we get:

$$
\mathbf{a}_i\mathbf{a}_i^\top \mathbf{a}_i= \mathbf{a}_i \underbrace{\|\mathbf{a}_i\|_2^2}_{\lambda} = \lambda \mathbf{a}_i.
$$

We have thus identifyied an eigenvalue-eigenvector pair, now we just need to show that all other eigenvalues of $\mathbf{a}_i\mathbf{a}_i^\top$ are smaller than $\|\mathbf{a}_i\|_2^2$.
It is easy to see that $\operatorname{rank}(\mathbf{a}_i\mathbf{a}_i^\top) = 1$, because every column of this outer product lives in $\operatorname{span}\{\mathbf{a}_i\}$. Therefore, as a consequence of the spectral theorem:

$$
\operatorname{rank}(\mathbf{a}_i\mathbf{a}_i^\top) = \operatorname{rank}(\mathbf{\Lambda}) = \sum_{j=1}^p I(\lambda_j \ne 0).
$$

This implies that, if $\mathbf{a}_i \ne \mathbf{0}$, $\lambda = \|\mathbf{a}_i\|_2^2$ is the only non-zero eigenvalue, and therefore 

$$
\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^\top) = \left\| \mathbf{a}_i\right\|_2^2.
$$

Notice that if $\mathbf{a}_i = \mathbf{0}$, this result holds trivially because $\mathbf{a}_i\mathbf{a}_i^\top$ is a matrix of zeros. $\quad \square$

---

__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

---

***Solution.***

From the previous point we get that $\|\mathbf{a}_i\mathbf{a}_i^\top\| := \lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^\top) = \left\| \mathbf{a}_i\right\|_2^2$, where $\|\cdot\|$ denotes the spectral norm (Schatten $\infty$-norm). Using the hint we have that the softmax Jacobian $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ satisfies 

<a id="ref_hint"></a>
$$
\|\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top\|_{2\to2} = \|\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top\| \le 1. \tag{given hint}
$$

Furthermore, by the properties of the Kronecker product, the spectral norm of the Kronecker product of two matrices is simply the product of the spectral norms, thus 

$$
\|(\mathbf{a}_i\mathbf{a}_i^\top) \otimes \left(\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \right)\| = \|\mathbf{a}_i\mathbf{a}_i^\top\| \|\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \| \underbrace{\le}_{\text{using \textbf{(j-1)}}} \left\| \mathbf{a}_i\right\|_2^2.
$$

Now applying the triangular inequality with respect to the spectral norm 

$$
\begin{align*}
\lambda_{\max}\left(\nabla^2 f_\mu(\mathbf{X}) \right) = \|\nabla^2 f_\mu(\mathbf{X})\| = \left\|\sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC}\right\| &\le \left\|\sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big)\right\| + \left\|\mu \mathbf{I}_{pC}\right \| \le\\
&\le \sum_{i=1}^n \left\|(\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big)\right\| + \left\|\mu \mathbf{I}_{pC}\right \| \le \\
&\le \sum_{i=1}^n \left\| \mathbf{a}_i\right\|_2^2 + \mu \quad \square
\end{align*}
$$

---

[Proof (given hint).](#ref_hint) We already showed in **(h)** that $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \succeq 0$. By definition, this means that, for any $\mathbf{v}\in \mathbb{R}^C$: 

$$
\mathbf{v}^\top \left(\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top\right)\mathbf{v} \ge 0.
$$

By developing this product we get, for the first term 

$$
\operatorname{Diag}(\mathbf{p}_i)\mathbf{v} = 
\begin{pmatrix}
p_{i,1}v_1\\
p_{i,2}v_2\\
\vdots\\
p_{i,C}v_C
\end{pmatrix} \iff
\mathbf v^\top\operatorname{Diag}(\mathbf{p}_i)\mathbf{v} = \sum_{c=1}^C v_c(p_{i,c}v_c) = \sum_{c=1}^C p_{i,c}v_c^2,
$$

while the second term is 

$$
\mathbf{v}^\top \mathbf{p}_i\mathbf{p}_i^\top \mathbf{v} = \| \mathbf{p}_i^\top \mathbf{v} \|_2^2 \underbrace{=}_{\mathbf{p}_i^\top \mathbf{v}\text{ is a scalar}} (\mathbf{p}_i^\top \mathbf{v})^2 = \left(\sum_{c=1}^C p_{i,c}v_c\right)^2.
$$

Hence, we have that

$$
\mathbf{v}^\top \left(\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top\right)\mathbf{v} = \sum_{c=1}^C p_{i,c}v_c^2 - \left(\sum_{c=1}^C p_{i,c}v_c\right)^2 \le \sum_{c=1}^C p_{i,c}v_c^2 \le \sum_{c=1}^C v_c^2 = \mathbf{v}^\top \mathbf{I}_C \mathbf{v},
$$

and consequently 

$$
\mathbf{I}_C \succeq \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \iff \text{all the eigenvalues of }\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \text{ are less than or equal to 1.} \quad \square
$$

---

__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


---

***Solution.***

Since $f_\mu: \mathbb{R}^{p\times C} \to \mathbb{R}$ and it is twice differentiable, we know that it is $L$-smooth if and only if 

$$
\nabla^2 f_\mu(\mathbf{X}) \preceq L \mathbf{I}_{pC}\quad \forall\ \mathbf{X} \in \mathbb{R}^{p\times C}.
$$

This is equivalent to say that the symmetric matrix satisfies $L \mathbf{I}_{pC} - \nabla^2f_\mu(\mathbf{X}) \succeq 0$, hence all eigenvalues of this matrix are nonnegative. Consider a generic eigenvalue-eigenvector pair for the Hessian, then it is straightforward to see that 

$$
(L \mathbf{I}_{pC} - \nabla^2 f_\mu(\mathbf{X})) \mathbf{v} = L \mathbf{v} - \nabla^2 f_\mu(\mathbf{X}) \mathbf{v} = L \mathbf{v} - \lambda \mathbf{v} = (L - \lambda) \mathbf{v}
$$

$L \mathbf{I}_{pC} - \nabla^2f_\mu(\mathbf{X})$ and the Hessian share the same eigenvectors, and the eigenvalues of this matrix are of the form $L - \lambda$. Then, by positive semidefiniteness, an equivalent condition for $f_\mu$ to be $L$-smooth is 

$$
L \ge \lambda \quad \forall\ \lambda \text{ eigenvalue of } \nabla^2 f_\mu(\mathbf{X}) \iff \lambda_\text{max}\left(\nabla^2f_\mu(\mathbf{X}) \right) \le L.
$$

Exploiting what we proved in **(j-2)**:

$$
\lambda_{\max}\left(\nabla^2 f_\mu(\mathbf{X}) \right) \le \sum_{i=1}^n\|\mathbf{a}_i\|_2^2 + \mu =: L.
$$

Finally, it suffices to notice, from the definition of $\mathbf{A} = [\mathbf{a}_1\ \ldots\ \mathbf{a}_n]^\top$, that

$$
\|\mathbf{A}\|_F = \sqrt{\sum_{i=1}^n \sum_{j=1}^p A_{ij}^2} = \sqrt{\sum_{i=1}^n \|\mathbf{a}_i\|^2_2} \iff  \|\mathbf{A}\|_F^2 = \sum_{i=1}^n \|\mathbf{a}_i\|^2_2.
$$

Plugging in this result into the previous inequality:

$$
\lambda_{\max}\left(\nabla^2 f_\mu(\mathbf{X}) \right) \le \|\mathbf{A}\|_F^2 + \mu =: L. \quad \square
$$

---

__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

---

***Solution.***

For two discrete probability distributions $\mathbf{p}_i$ and $q_i$ over the same event space, the Kullback-Leibler divergence is defined as

$$
\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)=\sum _{c = 1}^C q(c\mid\mathbf{a}_i)\log \left(\frac{q(c\mid\mathbf{a}_i)}{p_{i, c}}\right) \quad \text{where} \quad p_{i,c} = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

We want to show that

$$
\arg\min_{\mathbf{X}} \left\{\frac{1}{n}\sum_{i = 1}^n \mathrm{KL}(q_i\,\|\,\mathbf{p}_i)\right\} = \arg\min_{\mathbf{X}} \mathbb{E}_{\mathbf{b} \sim q(\mathbf{c}\mid\mathbf{A})}\left[ \frac{1}{n}\sum_{i = 1}^n (-\log p_{i,b_i}) \right].
$$

Starting from the expectation (dropping the notation for the expectation with respect to $\mathbf{b}$ for simplicity) and using the definition of expected value of a discrete random variable we have that 

$$
\mathbb{E} \left[ \frac{1}{n} \sum_{i=1}^n (-\log p_{i, b_i}) \right] = \frac{1}{n} \sum_{i=1}^n \mathbb{E} \left[ -\log p_{i, b_i} \right] = \frac{1}{n} \sum_{i=1}^n \left\{\sum_{c=1}^C q(c \mid \mathbf{a}_i) (-\log p_{i, c})\right\}.
$$

Developing now the average KL divergence we have that 

$$
\frac{1}{n}\sum_{i = 1}^n \mathrm{KL}(q_i\,\|\,\mathbf{p}_i) = \frac{1}{n}\sum_{i = 1}^n \left\{\sum _{c = 1}^C q(c\mid\mathbf{a}_i)\log \left(\frac{q(c\mid\mathbf{a}_i)}{p_{i, c}}\right)\right\} = \frac{1}{n}\sum_{i = 1}^n \left\{\sum _{c = 1}^C  q(c\mid\mathbf{a}_i) \left[\log q(c\mid\mathbf{a}_i) - \log p_{i, c} \right]\right\},
$$

and noticing that in this expression the sum can be split into two separate double sums, of which only the second one depends on $\mathbf{p}_i$ and hence on $\mathbf{X}$, we get that 

$$
\arg\min_{\mathbf{X}} \left\{\frac{1}{n}\sum_{i = 1}^n \mathrm{KL}(q_i\,\|\,\mathbf{p}_i)\right\} = \arg\min_{\mathbf{X}}\frac{1}{n}\sum_{i = 1}^n \left\{\sum _{c = 1}^C  q(c\mid\mathbf{a}_i) \left[ - \log p_{i, c} \right]\right\},
$$

which is exactly the objective function to minimise that we obtained earlier from the expected average negative log-likelihood. $\quad \square$

From **(a)**, we have that 

$$
\frac{f(\mathbf{X})}{n} = \frac{1}{n}\sum_{i=1}^n \left[ -\log p_{i,b_i} \right].
$$

As we saw earlier, by definition of expected value with respect to the true conditional label distribution $q_i$, $\mathbb{E}\left[ -\log p_{i,b_i} \right] = \sum_{c=1}^C q(c \mid \mathbf{a}_i) (-\log p_{i, c})$, thus

$$
\mathbb{E}\left[{\frac{f(\mathbf{X})}{n}}\right] = \frac{1}{n} \sum_{i=1}^n \left\{\sum_{c=1}^C q(c \mid \mathbf{a}_i) (-\log p_{i, c})\right\}.
$$

This makes $f(\mathbf{X})/n$ an unbiased estimator for the expected average NLL, which we cannot directly compute because of the fact that the true conditional label distribution is unknown.

---

The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


---

***Solution.***

We can rewrite the negative log-likelihood as 

$$
f(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big) = \sum_{i=1}^n \log\left( \frac{1}{\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})} \right) = \sum_{i=1}^n [- \log\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})].
$$

Taking the gradient and applying the chain rule 

$$
\begin{align*}
\nabla f(\mathbf{x}) = \sum_{i=1}^n \nabla[- \log\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})] &= \sum_{i=1}^n \left[-\frac{1}{\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})}\sigma'(b_i\, \mathbf{a}_i^\top \mathbf{x}) \nabla(b_i\, \mathbf{a}_i^\top \mathbf{x}) \right] =\\
&= \sum_{i=1}^n \left[-\frac{1}{\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})}\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})(1-\sigma(b_i\, \mathbf{a}_i^\top \mathbf{x})) b_i\, \mathbf{a}_i \right]= \quad \text{(using the hint)}\\
&= \sum_{i=1}^n \left[-\left(1-\frac{1}{1+\exp(-b_i\, \mathbf{a}_i^\top \mathbf{x})}\right) b_i\, \mathbf{a}_i \right] = \\
&= \sum_{i=1}^n \left[-\left(\frac{\exp(-b_i\, \mathbf{a}_i^\top \mathbf{x})}{1+\exp(-b_i\, \mathbf{a}_i^\top \mathbf{x})}\right) b_i\, \mathbf{a}_i \right]. 
\end{align*}
$$

Simply multiplying both the numerator and the denominator of the expression in the round parentheses by $\exp(b_i\, \mathbf{a}_i^\top \mathbf{x})$ we get: 

$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \left[-\sigma(-b_i \mathbf{a}_i^\top \mathbf{x}) b_i\, \mathbf{a}_i \right]. \quad \square
$$